# Open data from a web API: request, paginate, validate, test

*Session 2, block 2 practice. Author: course team, licence CC-BY-4.0.*

This notebook requests descriptions of Berlin datasets from **GovData**, the German national open-data portal, through its CKAN API. It shows each step that the workspace class `OpenDataClient` hides: the raw request, the JSON structure, pagination, parsing into a validated class, and an offline test with a mock transport.

**Needs:** internet access for sections 1–3; `httpx` and `pandas`. Start from the repository root, for example
`uv run --with httpx --with pandas --with jupyterlab jupyter lab`.

In [ ]:
import sys
from pathlib import Path

import httpx
import pandas as pd

# make the workspace package importable (uv run inside the workspace does this for you)
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "case-study").exists())
sys.path.insert(0, str(root / "sessions" / "02-software-engineering" / "workspace" / "src"))

from btitools.errors import ValidationError
from btitools.opendata import DatasetRecord, OpenDataClient

BASE = "https://ckan.govdata.de/api/3/action/"
HEADERS = {"User-Agent": "htw-spp-course-notebook/0.1 (teaching example)"}

## 1 · One request, looked at closely

A `GET` request to the endpoint `package_search` with three **query parameters**: the search text `q`, the page size `rows` and a filter `fq` for datasets published by Berlin.

In [ ]:
response = httpx.get(
    BASE + "package_search",
    params={"q": "baum", "rows": 2, "fq": "organization:berlin-open-data"},
    headers=HEADERS,
    timeout=10,
)
print(response.request.method, response.request.url)
print(response.status_code, response.headers.get("content-type"))

In [ ]:
response.raise_for_status()          # stop here if the status is 4xx or 5xx
payload = response.json()             # JSON text -> Python dicts and lists
print(payload.keys())
print("success:", payload["success"], "| total matches:", payload["result"]["count"])

In [ ]:
first = payload["result"]["results"][0]
# a CKAN "package" has many fields; we need only a few
{key: first[key] for key in ["name", "title", "metadata_modified", "num_resources"]}

**Your turn.** Change `q` to a topic of your team project (for example `"radverkehr"`, `"luftqualität"`, `"wohnungsbau"`). How many Berlin datasets match? Which fields of `first["resources"][0]` tell you the file format and download URL?

In [ ]:
# your code here


## 2 · Pagination

The response contains only `rows` results, but states the total `count`. To get all results, increase the offset `start` by the page size until `start >= count`. The loop below also stops after `max_records` to stay polite to the server.

In [ ]:
def fetch_all(query: str, page_size: int = 20, max_records: int = 60) -> list[dict]:
    packages, start = [], 0
    with httpx.Client(base_url=BASE, headers=HEADERS, timeout=10) as client:
        while len(packages) < max_records:
            params = {"q": query, "rows": page_size, "start": start,
                      "fq": "organization:berlin-open-data"}
            result = client.get("package_search", params=params).raise_for_status().json()["result"]
            packages += result["results"]
            print(f"start={start:>3}: {len(result['results'])} results (count={result['count']})")
            start += page_size
            if start >= result["count"] or not result["results"]:
                break
    return packages[:max_records]


packages = fetch_all("", page_size=20, max_records=60)
len(packages)

## 3 · Parse into a validated class and analyse

`DatasetRecord.from_ckan` (workspace, `src/btitools/opendata.py`) checks each package and cleans its fields. Invalid packages raise `ValidationError`; we count them instead of stopping.

In [ ]:
records, problems = [], []
for package in packages:
    try:
        records.append(DatasetRecord.from_ckan(package))
    except ValidationError as err:
        problems.append((package.get("name"), str(err)))

print(len(records), "valid,", len(problems), "invalid")
records[0]

In [ ]:
table = pd.DataFrame([{"title": r.title, "modified": r.modified, "n_resources": r.n_resources,
                       "n_tags": len(r.tags)} for r in records])
table.sort_values("modified", ascending=False).head()

In [ ]:
# The most frequent tags among these datasets
pd.Series([tag for r in records for tag in r.tags]).value_counts().head(10)

The workspace class does the same in one call (and retries after HTTP 429):

In [ ]:
with OpenDataClient() as client:
    page = client.search("baum", rows=5)
print(page.count, "matches;", [r.title for r in page.records][:3])

## 4 · Test the parsing without a network

A test should not depend on the portal being reachable or on its content today. `httpx.MockTransport` replaces the network with a function that plays the server. Here it returns one valid and one broken package; the client must keep the valid one and count the other.

In [ ]:
def fake_server(request: httpx.Request) -> httpx.Response:
    good = {"name": "strassenbaeume", "title": "Straßenbäume", "num_resources": 1,
            "metadata_modified": "2026-01-22T08:12:27", "organization": {"name": "berlin-open-data"}}
    broken = {**good, "name": "broken", "metadata_modified": "yesterday"}
    return httpx.Response(200, json={"success": True, "result": {"count": 2, "results": [good, broken]}})


with OpenDataClient(transport=httpx.MockTransport(fake_server)) as client:
    page = client.search("anything")

assert [r.name for r in page.records] == ["strassenbaeume"]
assert page.skipped == 1
print("parsing test passed:", page)

**Your turn.** Write a second fake server that answers `HTTP 503` and check with `try`/`except` that `client.search()` raises `btitools.errors.APIError`. Then move both checks into `tests/test_opendata.py` of the workspace as pytest functions; the existing tests there show the pattern.

In [ ]:
# your code here


## Data source and terms

Dataset descriptions: GovData (https://www.govdata.de), metadata of the organisation *Berlin Open Data*. The licence of each dataset is stated in its metadata and resources; check it before you use a dataset in your project (see the GovData terms of use at https://www.govdata.de).